In [1]:
import pandas as pd
import pdal
import json
import geopandas as gpd
import numpy as np
import glob
import sys
import os
import shutil
# from multiprocessing import Pool
from multiprocessing.dummy import Pool
import numpy as np

In [2]:
gdf_articulacao = gpd.read_file('results/folhas_sp_cortada.gpkg')

In [3]:
gdf_articulacao.to_crs(epsg=31983, inplace=True)

In [4]:
DATA_DIR_2024 = '/Users/fernandogomes/dev/LiDAR-Sampa-2024'
RESULT_FOLDER = '/Users/fernandogomes/dev/LiDAR_produtos'

In [5]:
class Scm:

    def __init__(self, scm) -> None:
        
        coords = [[xy[0], xy[1]] for xy in gdf_articulacao.set_index("nome").loc[scm].geometry.exterior.coords]
        xy_max = np.max(np.array(coords), axis=0) 
        xy_min = np.min(np.array(coords), axis=0)
        width, height = np.ceil(xy_max * 2) - np.ceil(xy_min * 2)

        origin_x, origin_y = np.floor(xy_min * 2)/2
        
        self.scm = scm
        self.width = width
        self.height = height
        self.origin_x = origin_x
        self.origin_y = origin_y

In [6]:
def pipeline(scm, ano):
    # Retorna o json com o Pipeline para o determinado SCM
    scm_att = Scm(scm)
    pipeline = [
        {
            "type": "readers.las",
            "filename": f'temp/{ano}-{scm}.laz',
            "override_srs": "EPSG:31983"
        },
        {
            "filename":f"{RESULT_FOLDER}/{ano}/MDS/MDS-{scm}-{ano}.tiff",
            "gdaldriver":"GTiff",
            "output_type":"max",
            "resolution":"0.5",
            "type": "writers.gdal",
            "gdalopts":"COMPRESS=ZSTD, PREDICTOR=3, BIGTIFF=YES",
            "width": scm_att.width,
            "height": scm_att.height,
            "origin_x": scm_att.origin_x,
            "origin_y": scm_att.origin_y,
            "default_srs": "EPSG:31983"
        },
        {
            "filename":f"{RESULT_FOLDER}/{ano}/MDT-points/MDT-points-{scm}-{ano}.tiff",
            "gdaldriver":"GTiff",
            "output_type":"max",
            "resolution":"0.5",
            "type": "writers.gdal",
            "gdalopts":"COMPRESS=ZSTD, PREDICTOR=3, BIGTIFF=YES",
            "width": scm_att.width,
            "height": scm_att.height,
            "origin_x": scm_att.origin_x,
            "origin_y": scm_att.origin_y,
            "default_srs": "EPSG:31983",
            "where": "(Classification == 2)"
        },
        {
            "type": "filters.delaunay",
            "where": "(Classification == 2)"
        },
        {
            "type": "filters.faceraster",
            "resolution": 0.5,
            "width": scm_att.width,
            "height": scm_att.height,
            "origin_x": scm_att.origin_x,
            "origin_y": scm_att.origin_y,
        },
        {
            "filename":f"{RESULT_FOLDER}/{ano}/MDT/MDT-{scm}-{ano}.tiff",
            "type": "writers.raster",
            # "filename":f"results/DTM-{grid_number}.tiff",
            "gdaldriver":"GTiff",
            "data_type": "float32",
            "gdalopts":"COMPRESS=ZSTD, PREDICTOR=3, BIGTIFF=YES",
            "nodata":"0",
        },
        {
            "type":"filters.range",
            "limits":"Classification[3:6]"
        },
        {
            "type":"filters.hag_dem",
            "raster": f"{RESULT_FOLDER}/{ano}/MDT/MDT-{scm}-{ano}.tiff",
            "zero_ground": True
        },
        {
            "type":"filters.ferry",
            "dimensions":"HeightAboveGround => Z"
        },
        {
            "type":"filters.range",
            "limits":"Z[0:300]"
        },
        # {
        #     "filename":f"results/{ano}/BHM/BHM-{scm}-{ano}-1m.tiff",
        #     "gdaldriver":"GTiff",
        #     "output_type":"max",
        #     "resolution":"1",
        #     "type": "writers.gdal",
        #     "gdalopts":"COMPRESS=ZSTD, PREDICTOR=3, BIGTIFF=YES",
        #     "width": scm_att.width,
        #     "height": scm_att.height,
        #     "origin_x": scm_att.origin_x,
        #     "origin_y": scm_att.origin_y,
        #     "nodata":"0",
        #     "data_type": "float32",
        #     "where": "(Classification == 6)",
        #     "default_srs": "EPSG:31983"
        # },
        {
            "filename":f"{RESULT_FOLDER}/{ano}/BHM/BHM-{scm}-{ano}-50cm.tiff",
            "gdaldriver":"GTiff",
            "output_type":"max",
            "resolution":"0.5",
            "type": "writers.gdal",
            "gdalopts":"COMPRESS=ZSTD, PREDICTOR=3, BIGTIFF=YES",
            "width": scm_att.width,
            "height": scm_att.height,
            "origin_x": scm_att.origin_x,
            "origin_y": scm_att.origin_y,
            "nodata":"0",
            "data_type": "float32",
            "where": "(Classification == 6)",
            "default_srs": "EPSG:31983"
        },
        # {
        #     "filename":f"results/{ano}/VHM/VHM-{scm}-{ano}-1m.tiff",
        #     "gdaldriver":"GTiff",
        #     "output_type":"max",
        #     "resolution":"1",
        #     "type": "writers.gdal",
        #     "gdalopts":"COMPRESS=ZSTD, PREDICTOR=3, BIGTIFF=YES",
        #     "width": scm_att.width,
        #     "height": scm_att.height,
        #     "origin_x": scm_att.origin_x,
        #     "origin_y": scm_att.origin_y,
        #     "nodata":"0",
        #     "data_type": "float32",
        #     "where": "(Classification == 3 || Classification == 4 || Classification == 5)",
        #     "default_srs": "EPSG:31983"
        # },
        {
            "filename":f"{RESULT_FOLDER}/{ano}/VHM/VHM-{scm}-{ano}-50cm.tiff",
            "gdaldriver":"GTiff",
            "output_type":"max",
            "resolution":"0.5",
            "type": "writers.gdal",
            "gdalopts":"COMPRESS=ZSTD, PREDICTOR=3, BIGTIFF=YES",
            "width": scm_att.width,
            "height": scm_att.height,
            "origin_x": scm_att.origin_x,
            "origin_y": scm_att.origin_y,
            "nodata":"0",
            "data_type": "float32",
            "where": "(Classification == 3 || Classification == 4 || Classification == 5)",
            "default_srs": "EPSG:31983"
        },
        {
            "type": "filters.range",
            "limits": "Classification[6:6]"
        },
        {
            "type":"filters.voxeldownsize",
            "cell":0.5,
            "mode":"center"
        },
        {
            "type":"writers.las",
            "filename":f"{RESULT_FOLDER}/{ano}/LiDAR/buildings-{scm}-{ano}-50cm.laz",
            "compression":"laszip"
        }
    ]
    return pipeline

In [7]:
def processo(scm):
    # Verifica se o processamento já foi realizado para o SCM
    if len(glob.glob(f"{RESULT_FOLDER}/2024/LiDAR/buildings-{scm}-2020-50cm.laz")) > 0:
        # print(f'SCM {scm} processado anteriormente')
        return None

    # Copia arquivos de 2017 e 2020 para uma pasta temporária
    file_2024 = glob.glob(f'{DATA_DIR_2024}/*{scm}*.laz')
    if len(file_2024) != 1:
        raise ValueError(f'Os arquivos do {scm} parecem não conforme!')
    
    print(f'Processando {scm}')

    shutil.copy(file_2024[0], f'temp/2024-{scm}.laz')
  
    # Processa o PDAL para cada ano: MDT, MDS, BHM, VHM
    mdt_mds = pdal.Pipeline(json.dumps(pipeline(scm, 2024)))
    n_points = mdt_mds.execute()
    # print(f'Executando MDT/MDS com {n_points} pontos')

    # Exclui os arquivos da pasta temporária
    os.remove(f'temp/2024-{scm}.laz')
    
    print(f'Processado {scm}')
    
    return None

In [8]:
def processa_tudo():
    # Itera sobre todos os SCMs
    # Utilizando multiprocessamento
    scms = gdf_articulacao.loc[:, 'nome'].to_list()
    with Pool(12) as p:
        _ = p.starmap(processo, zip(scms))
    # for scm in scms:
    #     processo(scm)
    return None

In [9]:
processa_tudo()

Processando Y-C-VI-3-SE-B-III-2
Processando Y-C-VI-4-SO-A-I-5
Processando Y-C-III-3-SE-C-II-2
Processando Y-C-VI-3-SE-D-I-3
Processando Y-C-VI-1-SE-D-IV-1
Processando Y-C-VI-3-SE-B-IV-5
Processando Y-C-III-3-SE-D-I-6
Processando Y-C-VI-4-SO-A-III-1
Processando Y-C-VI-1-NE-F-II-6
Processando Y-C-VI-1-NE-F-II-2
Processando Y-C-VI-3-SE-B-IV-6
Processando Y-C-VI-2-NO-E-I-1
Processado Y-C-VI-1-SE-D-IV-1
Processando Y-C-VI-1-SE-D-II-4
Processado Y-C-VI-1-NE-F-II-2
Processando Y-C-VI-1-NE-D-IV-5
Processado Y-C-III-3-SE-D-I-6
Processando Y-C-III-3-SE-D-I-3
Processado Y-C-VI-3-SE-B-IV-6
Processando Y-C-VI-3-SE-B-IV-3
Processado Y-C-VI-1-NE-F-II-6
Processando Y-C-VI-1-NE-F-II-3
Processado Y-C-VI-3-SE-B-III-2
Processando Y-C-VI-3-SE-B-I-5
Processado Y-C-VI-3-SE-B-IV-5
Processando Y-C-VI-3-SE-B-IV-2
Processado Y-C-VI-2-NO-E-I-1
Processando Y-C-VI-2-NO-C-III-4
Processado Y-C-III-3-SE-C-II-2
Processando Y-C-III-3-SE-A-IV-5
Processado Y-C-VI-4-SO-A-III-1
Processando Y-C-VI-4-SO-A-I-4
Processado Y-C-V